# Week 48, Day 1: Tries

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w048d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
words = ["car", "cart", "care", "cat", "dog"]

root = {}
for w in words:
    node = root
    for ch in w:
        node = node.setdefault(ch, {})
    node["$"] = True                    # marks the end of a word

print(sorted(root))
print(sorted(root["c"]["a"]))
print(sorted(root["c"]["a"]["r"]))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}      # character -> TrieNode
        self.end = False        # does a word end here?

class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        for w in words:
            self.insert(w)

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.end = True

t = Trie(["car", "cart", "care", "cat", "dog"])
print(sorted(t.root.children), sorted(t.root.children["c"].children["a"].children))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        for w in words:
            node = self.root
            for ch in w:
                node = node.children.setdefault(ch, TrieNode())
            node.end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def contains(self, word):
        node = self._walk(word)
        return node is not None and node.end

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

t = Trie(["car", "cart", "care", "cat", "dog"])
print(t.contains("car"), t.contains("ca"), t.starts_with("ca"), t.starts_with("cab"), t.starts_with(""))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end, self.count = {}, False, 0

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        if self.contains(word):
            return                              # don't count a word twice
        node = self.root
        node.count += 1
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
            node.count += 1
        node.end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def contains(self, word):
        node = self._walk(word)
        return node is not None and node.end

    def count_prefix(self, prefix):
        node = self._walk(prefix)
        return 0 if node is None else node.count

t = Trie()
for w in ["car", "cart", "care", "cat", "dog", "car"]:
    t.insert(w)
print(t.count_prefix("car"), t.count_prefix("ca"), t.count_prefix(""), t.count_prefix("x"))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    return root

def with_prefix(root, prefix):
    node = root
    for ch in prefix:
        node = node.children.get(ch)
        if node is None:
            return []
    out, stack = [], [(node, prefix)]
    while stack:
        node, word = stack.pop()
        if node.end:
            out.append(word)
        for ch in sorted(node.children, reverse=True):
            stack.append((node.children[ch], word + ch))
    return out

root = build(["car", "cart", "care", "cat", "dog", "carton"])
print(with_prefix(root, "car"), with_prefix(root, "d"), with_prefix(root, "z"))

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    return root

def delete(root, word):
    path = [root]                         # path[i] is the node after i characters
    for ch in word:
        nxt = path[-1].children.get(ch)
        if nxt is None:
            return False
        path.append(nxt)
    if not path[-1].end:
        return False
    path[-1].end = False
    for i in range(len(word), 0, -1):     # prune from the bottom up
        node = path[i]
        if node.end or node.children:
            break
        del path[i - 1].children[word[i - 1]]
    return True

root = build(["car", "cart", "dog"])
print(delete(root, "dog"), sorted(root.children))
print(delete(root, "car"), sorted(root.children["c"].children["a"].children["r"].children))
print(delete(root, "ca"), delete(root, "cow"))

> **Key idea.** A trie stores strings as paths from the root, one character per edge, sharing common prefixes; each node has a dict of children and an end-of-word flag. Insert, contains and starts_with walk one node per character: O(L), independent of the number of words. Keep a count in each node (and don't count repeats) to count words by prefix in O(L). List words with a prefix by walking to its node and visiting the subtree in sorted order. Delete by un-marking the end, then pruning nodes that are neither ends nor have children, from the bottom up. Tries cost a lot of memory, so use a set unless prefixes matter.

### Exercise: A trie

Write `Trie` with `insert(word)`, `contains(word)`, `starts_with(prefix)` and `__len__` (the number of distinct words), using `TrieNode`s. No sets or lists of the words themselves.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False

class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, string\ndef test_trie():\n    \"words and prefixes\"\n    t = Trie([\"car\", \"cart\", \"care\", \"cat\", \"dog\", \"car\"])\n    assert len(t) == 5 and t.contains(\"car\") and not t.contains(\"ca\") and not t.contains(\"carts\")\n    assert t.starts_with(\"ca\") and t.starts_with(\"\") and not t.starts_with(\"cab\") and t.starts_with(\"dog\")\n    e = Trie()\n    assert len(e) == 0 and not e.contains(\"\") and e.starts_with(\"\")\n    e.insert(\"\")\n    assert e.contains(\"\") and len(e) == 1\ndef test_random():\n    \"against a set\"\n    rng = random.Random(1)\n    words = {\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 6))) for _ in range(300)}\n    t = Trie(words)\n    assert len(t) == len(words)\n    for _ in range(500):\n        s = \"\".join(rng.choice(\"abcd\") for _ in range(rng.randint(0, 6)))\n        assert t.contains(s) == (s in words) and t.starts_with(s) == any(w.startswith(s) for w in words)\n    assert all(not isinstance(v, (set, list)) for v in vars(t).values())")

### Exercise: Counting by prefix

Write `CountingTrie` where each node keeps `count` (the number of distinct words at or below it) and `times` (how many times the word ending exactly there has been inserted). Methods: `insert(word)`, `count_prefix(prefix)` (distinct words starting with it), `frequency(word)` (times inserted; 0 if never) and `total()` (all insertions including repeats). Each must be O(length), with no walks over subtrees.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.times = 0

class CountingTrie:
    def __init__(self):
        self.root = TrieNode()
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import Counter\ndef test_counts():\n    \"counts\"\n    t = CountingTrie()\n    for w in [\"car\", \"cart\", \"care\", \"cat\", \"dog\", \"car\", \"car\"]:\n        t.insert(w)\n    assert t.count_prefix(\"car\") == 3 and t.count_prefix(\"ca\") == 4 and t.count_prefix(\"\") == 5 and t.count_prefix(\"x\") == 0\n    assert t.frequency(\"car\") == 3 and t.frequency(\"ca\") == 0 and t.frequency(\"zebra\") == 0 and t.total() == 7\ndef test_random():\n    \"against a Counter\"\n    rng = random.Random(2)\n    t, ref = CountingTrie(), Counter()\n    for _ in range(2000):\n        w = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 5)))\n        t.insert(w)\n        ref[w] += 1\n    for _ in range(300):\n        p = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 5)))\n        assert t.count_prefix(p) == sum(1 for w in ref if w.startswith(p)) and t.frequency(p) == ref[p]\n    assert t.total() == 2000")

### Exercise: Listing by prefix

Write `with_prefix(root, prefix, limit=None)`: the words starting with `prefix`, alphabetically, stopping after `limit` words if a limit is given (and not visiting the rest of the subtree). Use an explicit stack, not recursion. Then `all_words(root)` and `longest_word(root)`: the longest word (alphabetically first on a tie; `None` if the trie has no words). A `build(words)` helper is provided.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    return root

def with_prefix(root, prefix, limit=None):
    ___

def all_words(root):
    ___

def longest_word(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_listing():\n    \"listing\"\n    root = build([\"car\", \"cart\", \"care\", \"cat\", \"dog\", \"carton\", \"a\"])\n    assert with_prefix(root, \"car\") == [\"car\", \"care\", \"cart\", \"carton\"] and with_prefix(root, \"z\") == []\n    assert with_prefix(root, \"ca\", limit=2) == [\"car\", \"care\"] and with_prefix(root, \"\", limit=0) == []\n    assert all_words(root) == [\"a\", \"car\", \"care\", \"cart\", \"carton\", \"cat\", \"dog\"]\n    assert longest_word(root) == \"carton\" and longest_word(build([])) is None and longest_word(build([\"ab\", \"aa\"])) == \"aa\"\ndef test_random():\n    \"against sorting\"\n    rng = random.Random(3)\n    words = {\"\".join(rng.choice(\"abcde\") for _ in range(rng.randint(1, 7))) for _ in range(400)}\n    root = build(words)\n    assert all_words(root) == sorted(words)\n    for p in [\"\", \"a\", \"ab\", \"eee\", \"ca\"]:\n        assert with_prefix(root, p, limit=5) == sorted(w for w in words if w.startswith(p))[:5]\ndef test_deep():\n    \"a very long word\"\n    root = build([\"x\" * 5000])\n    assert with_prefix(root, \"xx\") == [\"x\" * 5000] and longest_word(root) == \"x\" * 5000")

### Exercise: Deleting cleanly

Write `delete(root, word)`: remove the word if present (returning `True`, else `False`), pruning every node that is no longer an end and has no children, from the bottom up, without recursion. Then `node_count(root)`: the number of nodes in the trie including the root (with a stack). After deleting every word, only the root should remain.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    return root

def delete(root, word):
    ___

def node_count(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef words_in(root):\n    out, stack = [], [(root, \"\")]\n    while stack:\n        n, w = stack.pop()\n        if n.end:\n            out.append(w)\n        stack.extend((c, w + ch) for ch, c in n.children.items())\n    return sorted(out)\ndef test_delete():\n    \"deleting\"\n    root = build([\"car\", \"cart\", \"dog\"])\n    assert node_count(root) == 8\n    assert delete(root, \"dog\") and node_count(root) == 5 and words_in(root) == [\"car\", \"cart\"]\n    assert delete(root, \"car\") and node_count(root) == 5 and words_in(root) == [\"cart\"]\n    assert not delete(root, \"car\") and not delete(root, \"ca\") and not delete(root, \"cartoon\")\n    assert delete(root, \"cart\") and node_count(root) == 1\ndef test_random():\n    \"random deletions\"\n    rng = random.Random(4)\n    words = list({\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 6))) for _ in range(200)})\n    root = build(words)\n    rng.shuffle(words)\n    live = set(words)\n    for w in words:\n        assert delete(root, w)\n        live.discard(w)\n        assert words_in(root) == sorted(live)\n        assert node_count(root) == 1 + len({w2[:i] for w2 in live for i in range(1, len(w2) + 1)})\n    assert node_count(root) == 1")

### Exercise: Shared beginnings

Write `longest_common_prefix(words)`: the longest prefix shared by **all** the words (`""` for an empty list), by building a trie and walking down from the root while there's exactly one child and no word has ended. Then `shortest_unique_prefixes(words)`: for each word (distinct words, none a prefix of another), the shortest prefix that no other word starts with, using counts in the trie.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False
        self.count = 0

def longest_common_prefix(words):
    ___

def shortest_unique_prefixes(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, os\ndef test_lcp():\n    \"common prefixes\"\n    assert longest_common_prefix([\"flower\", \"flow\", \"flight\"]) == \"fl\"\n    assert longest_common_prefix([\"dog\", \"car\"]) == \"\" and longest_common_prefix([]) == \"\" and longest_common_prefix([\"solo\"]) == \"solo\"\n    assert longest_common_prefix([\"ab\", \"abc\", \"ab\"]) == \"ab\"\n    rng = random.Random(5)\n    for _ in range(100):\n        ws = [\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 5))) for _ in range(rng.randint(1, 5))]\n        assert longest_common_prefix(ws) == os.path.commonprefix(ws)\ndef test_unique():\n    \"unique prefixes\"\n    assert shortest_unique_prefixes([\"zebra\", \"dog\", \"duck\", \"dove\"]) == [\"z\", \"dog\", \"du\", \"dov\"]\n    assert shortest_unique_prefixes([\"apple\"]) == [\"a\"]")

### Bug hunt: Every prefix is a word

This `contains` checks only that the walk succeeds, not that a word ends there, so it treats every prefix as a word: this prints `True True`. Check the end mark so it prints `True False`.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

root = TrieNode()
for w in ["cart", "dog"]:
    node = root
    for ch in w:
        node = node.children.setdefault(ch, TrieNode())
    node.end = True

def contains(word):
    node = root
    for ch in word:
        node = node.children.get(ch)
        if node is None:
            return False
    return True

print(contains("cart"), contains("car"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check the end\"\n    expect_output(\"True False\")")

### Bug hunt: Starting a new branch every time

This insert makes a **new** child for every character, even when one already exists, so each word replaces the branch of the word before it: after inserting "car" and "cat", "car" is gone and this prints `False True`. Reuse an existing child so it prints `True True`.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

def insert(root, word):
    node = root
    for ch in word:
        node.children[ch] = TrieNode()
        node = node.children[ch]
    node.end = True

def contains(root, word):
    node = root
    for ch in word:
        node = node.children.get(ch)
        if node is None:
            return False
    return node.end

root = TrieNode()
insert(root, "car")
insert(root, "cat")
print(contains(root, "car"), contains(root, "cat"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reuse the branch\"\n    expect_output(\"True True\")")

### Bug hunt: Pruning too far

This delete prunes every node on the path that has no **other** children, without checking whether a shorter word ends there, so deleting "cart" also deletes "car": this prints `False`. Stop pruning at a node that is the end of a word, so it prints `True`.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children, self.end = {}, False

def build(words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True
    return root

def delete(root, word):
    path = [root]
    for ch in word:
        path.append(path[-1].children[ch])
    path[-1].end = False
    for i in range(len(word), 0, -1):
        if path[i].children:
            break
        del path[i - 1].children[word[i - 1]]

def contains(root, word):
    node = root
    for ch in word:
        node = node.children.get(ch)
        if node is None:
            return False
    return node.end

root = build(["car", "cart"])
delete(root, "cart")
print(contains(root, "car"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep shorter words\"\n    expect_output(\"True\")")

### Puzzle: Inserting into a trie

Arrange the lines into a trie insert. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# A trie holding exactly: "tea", "ten", "to", "inn", "in", "tent".

p = ___   # how many children the root has
q = ___   # how many nodes the trie has, counting the root
r = ___   # the words starting with "te", alphabetically
s = ___   # True or False: is "te" a word in the trie?
t = ___   # how many words start with "in"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 10 and r == [\"tea\", \"ten\", \"tent\"], \"p, q and r\"\n    assert s is False and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: wildcards and replacements

Write `WordDictionary` with:

- `add(word)`;
- `search(pattern)`: whether any word matches `pattern` exactly, where `.` in the pattern matches any one character. At a `.`, every child must be tried, so search with an explicit stack of `(node, position)` pairs;
- `matches(pattern)`: every matching word, alphabetically;
- `replace_roots(sentence)`: replace each word of a sentence (split on spaces) by the **shortest** stored word that is a prefix of it, if any, joining the result with spaces. For example, with roots "cat", "bat" and "rat", "the cattle was rattled by the battery" becomes "the cat was rat by the bat". Walk down the trie along the word and stop at the first end mark.

The checks compare against a brute-force version on random words and patterns.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = False

class WordDictionary:
    def __init__(self):
        self.root = TrieNode()
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, re\ndef test_examples():\n    \"examples\"\n    d = WordDictionary()\n    for w in [\"bad\", \"dad\", \"mad\", \"cat\", \"bat\", \"rat\"]:\n        d.add(w)\n    assert d.search(\"bad\") and not d.search(\"pad\") and d.search(\".ad\") and d.search(\"b..\") and not d.search(\"b...\") and not d.search(\"\")\n    assert d.matches(\".at\") == [\"bat\", \"cat\", \"rat\"] and d.matches(\"...\") == [\"bad\", \"bat\", \"cat\", \"dad\", \"mad\", \"rat\"] and d.matches(\"x..\") == []\n    assert d.replace_roots(\"the cattle was rattled by the battery\") == \"the cat was rat by the bat\"\n    d.add(\"ca\")\n    assert d.replace_roots(\"cattle\") == \"ca\" and d.replace_roots(\"\") == \"\"\ndef test_random():\n    \"against brute force\"\n    rng = random.Random(6)\n    words = {\"\".join(rng.choice(\"abc\") for _ in range(rng.randint(1, 5))) for _ in range(60)}\n    d = WordDictionary()\n    for w in words:\n        d.add(w)\n    for _ in range(300):\n        p = \"\".join(rng.choice(\"abc.\") for _ in range(rng.randint(1, 5)))\n        want = sorted(w for w in words if re.fullmatch(p, w))\n        assert d.matches(p) == want and d.search(p) == bool(want), p")

## Warm-up

Five words, but only two branches from the root: everything starting with "c" shares one path, and "car", "cart" and "care" share "car". The `"$"` key marks where a word ends, so "car" is a word even though the path carries on.

## A tree of characters

A **trie** (from re*trie*val; Edward Fredkin named it in 1960, after René de la Briandais described the idea in 1959) is a tree where:

- each node has up to one child per character, kept in a **dict** from character to child node;
- the path from the root to a node spells a **prefix**;
- each node has a flag saying whether that prefix is a **whole word**.

```text
            (root)
           /      \
          c        d
          |        |
          a        o
        /   \      |
       r*    t*    g*          * = end of a word
      / \
     t*  e*                    words: car, cart, care, cat, dog
```

A class makes it clearer than nested dicts (and leaves room for extra fields later):

Inserting a word of length L walks or creates L nodes: **O(L)**, however many words the trie already holds.

## Words and prefixes

Looking something up walks down the same way, one character at a time. Two different questions, one walk:

- `contains(word)`: walk the word's characters; it's a word only if the walk succeeds **and** the last node is marked as an end.
- `starts_with(prefix)`: walk the prefix; any successful walk means some word starts with it.

Both are O(L) for a word of length L. A shared helper that returns the node at the end of a walk (or `None`) keeps them short:

The empty prefix walks nowhere and stops at the root, so every trie "starts with" it, which is the right answer: every word begins with the empty string.

## Counting

How many words start with "car"? Walking to the "car" node and counting the ends below it costs time proportional to that whole subtree. Instead, keep a **count** in every node: the number of words passing through it (or ending at it). Increase every count on the path when inserting a new word, and counting by prefix becomes a single O(L) walk.

Notice the check at the start of `insert`: without it, inserting "car" twice would count it twice. (If you **want** duplicates counted, as in a word-frequency trie, keep a separate count of how many times each word ends at a node.)

## Listing and deleting

To **list** the words with a prefix, walk to the prefix's node, then visit its whole subtree, building each word as you go down. Visiting children in sorted order gives the words in alphabetical order. An explicit stack (Week 45 Day 3) avoids recursion; push children in **reverse** sorted order so the smallest is popped first:

Pre-order puts "car" before "care" and "cart", exactly as alphabetical order wants: a word comes before every longer word that starts with it.

**Deleting** a word means un-marking its end. That alone is correct, but it can leave a dead branch: delete "dog" and the nodes d, o, g lead to no word at all. To tidy up, remember the path, then walk back up from the end, removing each node that is neither an end nor has children:

## Tries, sets and sorted lists

| question | `set` | sorted list + `bisect` | trie |
|---|---|---|---|
| is w a word? | O(L) average (hashing w) | O(L log n) | O(L) |
| add a word | O(L) average | O(n) (shifting) | O(L) |
| how many words start with p? | O(n L) | O(L log n), two searches | O(L) with counts |
| list words starting with p | O(n L) | O(L log n + output) | O(L + size of that subtree) |
| memory | compact | compact | large: a dict per node |

The trie's weakness is memory: every node is an object with its own dict, and a Python dict is a few hundred bytes even when nearly empty. Real implementations save space by **compressing** chains of single-child nodes into one edge labelled with a whole string (a **radix tree**, used in network routers and some databases), or by using arrays instead of dicts for small alphabets.

When you only need "is this a word?", use a set. Reach for a trie when prefixes matter: autocomplete, spell-checkers, routing tables, word games, and matching many patterns at once.

## Practice

## Quiz

## Daily challenge

## Before you go

Draw the trie for the words in your own name and three friends' names, and count its nodes. How many would a radix tree need? Tomorrow: finding a pattern in a text, from the obvious method to Knuth, Morris and Pratt's, which never looks at a character of the text twice.